# FilBAO: BAO fit of the subsamples (plan step 3)

**Input:** the monopoles and the power-spectra file written by `scripts/multidark_analysis.py`
(in `data/processed/`). **Output:** one summary table per run, the Δα / ΔΣ differences, each
monopole with its fit, and the (α, Σ) contours. All definitions are in `src/bao_fitting.py`.

**Model.** $\xi(s) = B^2\big[\xi_{\rm nw}(s) + \xi_{\rm wig}(\alpha s;\Sigma)\big] + a_0 + a_1\,(100/s) + a_2\,(100/s)^2$,
with the BAO wiggle damped by $e^{-k^2\Sigma^2/2}$ and averaged over each $s$ bin like the data.

**Per sample** (`analyse_sample`):
1. bias $b$ from 40–70 $h^{-1}$Mpc (used only to build the covariance);
2. Gaussian covariance from the sample's **measured** $P(k)$ (`gaussian_cov_pk`);
3. two fits on the $(\alpha, \Sigma)$ grid, with $B^2$ and the broadband solved exactly:
   * **α**: Σ ~ N(theory = 7.67, 2), the same prior for every sample;
   * **Σ**: flat prior 0–16, α free and marginalised; its 2D posterior gives the contours;
4. detection: Δχ² = χ²(no-wiggle) − χ²(BAO), and **p** = fraction of no-wiggle Gaussian mocks with a larger Δχ².

**Per pair** (`difference_test`): Δα and ΔΣ, with errors from correlated mocks of all samples
of the run (they share the box, so their errors are correlated) and a null p-value.

**One run of `multidark_analysis.py` = one `run_all`**: all samples of a `run_all` must be in
the same spectra file.

In [ ]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))   # notebook lives in notebooks/final
sys.path.insert(0, os.path.join(REPO, 'src'))
import matplotlib.pyplot as plt
from bao_fitting import run_all

MONO = os.path.join(REPO, 'data', 'processed', 'monopoles', 'box')
PK = os.path.join(REPO, 'data', 'processed', 'pk', 'box')

## Run 1: Full, Filament, Non-filament and their random dilutions
`multidark_analysis.py` with `split_vars` = `dist_fil` only and `dilution_controls = 'all'`.
`binsep` must match `bin_size_2d` in the script. The `<bin_desc>` of each file is printed by
the script (`Processing: <label> -> <bin_desc>`).

In [ ]:
binsep = 5.0
base = os.path.join(MONO, f'mag=-21.2_sep=5.0-150.0_binsep={binsep:.1f}_')
samples = {
    'Full':         base + 'full.npz',
    'Filament':     base + '0.0-d_mathrmfilleq5.0.npz',
    'Non-filament': base + '10.0-d_mathrmfilleq100.0.npz',
    # random dilutions of Full with the same N as Filament / Non-filament (N from the script output):
    # 'Full dil nFil':    base + 'FullrandomN-<N_fil>r20261016.npz',
    # 'Full dil nNonFil': base + 'FullrandomN-<N_nonfil>r20261016.npz',
}
pk_file = os.path.join(PK, 'pk_XISMU_box_mag-21.2_dist_fil.npz')

# Comparisons to report (decide these before looking at the results)
pairs = [('Filament', 'Non-filament'), ('Filament', 'Full'), ('Non-filament', 'Full'),
         # ('Filament', 'Full dil nFil'), ('Non-filament', 'Full dil nNonFil'),
        ]
res = run_all(samples, pk_file, pairs=pairs, fit_range=(65, 135), n_mock=300, n_mock_diff=400,
              save_plots=False)

## Run 2: filament × local density (the four cells)
`multidark_analysis.py` with `split_vars` = `dist_fil` and `log_rho_3`, `dilution_controls = []`.
The cell file names contain the ρ₃ percentile values of the catalogue; copy them from the
script output.

In [ ]:
# samples_2x2 = {
#     'Full':              base + 'full.npz',
#     'Fil dense':         base + '<bin_desc>.npz',
#     'Fil underdense':    base + '<bin_desc>.npz',
#     'NonFil dense':      base + '<bin_desc>.npz',
#     'NonFil underdense': base + '<bin_desc>.npz',
# }
# pk_file_2x2 = os.path.join(PK, 'pk_XISMU_box_mag-21.2_dist_fillog_rho_3.npz')
# pairs_2x2 = [('Fil dense', 'NonFil dense'), ('Fil underdense', 'NonFil underdense')]
# res_2x2 = run_all(samples_2x2, pk_file_2x2, pairs=pairs_2x2, fit_range=(65, 135),
#                   n_mock=300, n_mock_diff=400, save_plots=False)

`save_plots=True` writes each fit (`bao_fit_<bin_desc>.png`) and the contours
(`alpha_sigma_contours.png`) to `figures/` (overwritten by the next run).
Error calibration of the fits (pull std, coverage) is in `notebooks/exploratory/calibration_mocks.ipynb`.